# Phone Active Screen Algorithm

This notebook evaluates an aggressive Android active-screen algorithm intended to minimize false positives.

Candidate definition:

```text
active = SCREEN_INTERACTIVE state
         AND KEYGUARD_HIDDEN state
         AND not between DEVICE_SHUTDOWN and DEVICE_STARTUP
```

Rules implemented here:

- sort by timestamp and original event id (`line_number` from the JSONL file);
- query events before the analysis window to establish initial state;
- start only when both required states are known;
- stop on the first screen-off, keyguard-shown, shutdown, or range-end boundary;
- treat repeated events as idempotent;
- report contradictory transitions as anomalies;
- report unknown time rather than guessing;
- calculate both state-machine and interval-intersection results as a consistency check.


In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import polars as pl

from timeline.domain.enums import EventKind
from timeline.ingest.android import import_android_unlock_jsonl, parse_android_timestamp


In [2]:
PATH = Path("../data/android/unlock-events.jsonl")

# Use None to analyze the full available log. Set either value to a naive local
# datetime to test window-boundary behavior.
ANALYSIS_START: datetime | None = None
ANALYSIS_END: datetime | None = None


def load_android_rows(path: Path) -> pl.DataFrame:
    rows = []
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            rows.append({
                "line_number": line_number,
                "timestamp": parse_android_timestamp(row["timestamp"]),
                "event": row["event"],
                "event_type": row.get("event_type"),
            })
    return pl.DataFrame(rows).sort(["timestamp", "line_number"])


df = load_android_rows(PATH)
window_start = ANALYSIS_START or df.select(pl.col("timestamp").min()).item()
window_end = ANALYSIS_END or df.select(pl.col("timestamp").max()).item()

df.select([
    pl.len().alias("rows"),
    pl.col("timestamp").min().alias("first_timestamp"),
    pl.col("timestamp").max().alias("last_timestamp"),
])


rows,first_timestamp,last_timestamp
u32,datetime[μs],datetime[μs]
2802,2026-07-26 18:15:27.403,2026-08-02 17:41:13.171


In [3]:
df.group_by(["event_type", "event"]).agg(pl.len().alias("count")).sort("count", descending=True)


event_type,event,count
i64,str,u32
15,"""SCREEN_INTERACTIVE""",760
16,"""SCREEN_NON_INTERACTIVE""",760
17,"""KEYGUARD_SHOWN""",637
18,"""KEYGUARD_HIDDEN""",635
27,"""DEVICE_STARTUP""",5
26,"""DEVICE_SHUTDOWN""",5


## Baseline: Existing Importer

The current importer compares against one event pair at a time. `keyguard` is the default production strategy right now; `screen` is useful context because the new algorithm intersects both states.


In [4]:
@dataclass(frozen=True)
class Interval:
    start: datetime
    end: datetime
    start_line: int | None
    end_line: int | None
    reason: str

    @property
    def seconds(self) -> float:
        return (self.end - self.start).total_seconds()


def importer_intervals(strategy: str) -> list[Interval]:
    result = import_android_unlock_jsonl(PATH, strategy=strategy)  # type: ignore[arg-type]
    intervals = []
    open_event = None
    for event in result.events:
        if event.event_kind == EventKind.INTERVAL_START:
            open_event = event
        elif open_event is not None:
            intervals.append(Interval(open_event.timestamp, event.timestamp, None, None, "existing_importer"))
            open_event = None
    return intervals


existing_keyguard = importer_intervals("keyguard")
existing_screen = importer_intervals("screen")
len(existing_keyguard), sum(i.seconds for i in existing_keyguard) / 3600, len(existing_screen), sum(i.seconds for i in existing_screen) / 3600


(634, 46.15026305555555, 757, 64.6809138888889)

## Shared Helpers


In [5]:
START_EVENTS = {"SCREEN_INTERACTIVE", "KEYGUARD_HIDDEN", "DEVICE_STARTUP"}
STOP_EVENTS = {"SCREEN_NON_INTERACTIVE", "KEYGUARD_SHOWN", "DEVICE_SHUTDOWN"}


def intervals_frame(intervals: list[Interval]) -> pl.DataFrame:
    return pl.DataFrame([
        {
            "start": interval.start,
            "end": interval.end,
            "seconds": interval.seconds,
            "minutes": interval.seconds / 60,
            "start_line": interval.start_line,
            "end_line": interval.end_line,
            "reason": interval.reason,
        }
        for interval in intervals
    ])


def summarize_intervals(name: str, intervals: list[Interval]) -> dict[str, object]:
    seconds = [interval.seconds for interval in intervals]
    return {
        "strategy": name,
        "intervals": len(intervals),
        "hours": sum(seconds) / 3600,
        "min_seconds": min(seconds) if seconds else None,
        "median_seconds": sorted(seconds)[len(seconds) // 2] if seconds else None,
        "max_seconds": max(seconds) if seconds else None,
    }


def clip_interval(start: datetime, end: datetime, window_start: datetime, window_end: datetime) -> tuple[datetime, datetime] | None:
    clipped_start = max(start, window_start)
    clipped_end = min(end, window_end)
    if clipped_start >= clipped_end:
        return None
    return clipped_start, clipped_end


## State-Machine Implementation

`None` means unknown. The machine reports unknown ranges instead of assuming screen/keyguard state. Repeated same-direction state events are recorded as idempotent anomalies but do not change intervals.


In [6]:
@dataclass
class PhoneState:
    screen_interactive: bool | None = None
    keyguard_hidden: bool | None = None
    shutdown_active: bool = False


def active_state(state: PhoneState) -> bool | None:
    if state.shutdown_active:
        return False
    if state.screen_interactive is None or state.keyguard_hidden is None:
        return None
    return state.screen_interactive and state.keyguard_hidden


def apply_event(state: PhoneState, event: str) -> tuple[str, bool | None, bool | None]:
    """Apply one event and return (anomaly_kind, before_value, after_value)."""
    if event == "SCREEN_INTERACTIVE":
        before = state.screen_interactive
        state.screen_interactive = True
        return ("repeated_screen_interactive" if before is True else "", before, True)
    if event == "SCREEN_NON_INTERACTIVE":
        before = state.screen_interactive
        state.screen_interactive = False
        return ("repeated_screen_non_interactive" if before is False else "", before, False)
    if event == "KEYGUARD_HIDDEN":
        before = state.keyguard_hidden
        state.keyguard_hidden = True
        return ("repeated_keyguard_hidden" if before is True else "", before, True)
    if event == "KEYGUARD_SHOWN":
        before = state.keyguard_hidden
        state.keyguard_hidden = False
        return ("repeated_keyguard_shown" if before is False else "", before, False)
    if event == "DEVICE_SHUTDOWN":
        before = state.shutdown_active
        state.shutdown_active = True
        return ("repeated_device_shutdown" if before is True else "", before, True)
    if event == "DEVICE_STARTUP":
        before = state.shutdown_active
        state.shutdown_active = False
        return ("startup_without_shutdown" if before is False else "", before, False)
    return "", None, None


def analyze_state_machine(df: pl.DataFrame, window_start: datetime, window_end: datetime) -> tuple[list[Interval], list[Interval], pl.DataFrame]:
    state = PhoneState()
    active_start: datetime | None = None
    active_start_line: int | None = None
    unknown_start: datetime | None = window_start
    intervals: list[Interval] = []
    unknown_ranges: list[Interval] = []
    anomalies: list[dict[str, object]] = []

    all_rows = df.filter(pl.col("timestamp") <= window_end).iter_rows(named=True)
    for row in all_rows:
        ts = row["timestamp"]
        event = row["event"]
        in_window = ts >= window_start

        before_active = active_state(state)
        anomaly, before_value, after_value = apply_event(state, event)
        after_active = active_state(state)

        if anomaly:
            anomalies.append({
                "timestamp": ts,
                "line_number": row["line_number"],
                "event": event,
                "anomaly": anomaly,
                "previous_value": before_value,
                "new_value": after_value,
                "inside_window": in_window,
            })

        # Events before the window are only used to establish initial state.
        if not in_window:
            continue

        if before_active is None and after_active is not None and unknown_start is not None:
            if unknown_start < ts:
                unknown_ranges.append(Interval(unknown_start, ts, None, row["line_number"], "state_unknown"))
            unknown_start = None
        elif after_active is None and unknown_start is None:
            unknown_start = ts

        if before_active is True and after_active is not True and active_start is not None:
            intervals.append(Interval(active_start, ts, active_start_line, row["line_number"], f"stopped_by_{event.lower()}"))
            active_start = None
            active_start_line = None
        elif before_active is not True and after_active is True:
            active_start = ts
            active_start_line = row["line_number"]

    final_active = active_state(state)
    if active_start is not None and active_start < window_end:
        intervals.append(Interval(active_start, window_end, active_start_line, None, "stopped_by_range_end"))
    if final_active is None and unknown_start is not None and unknown_start < window_end:
        unknown_ranges.append(Interval(unknown_start, window_end, None, None, "state_unknown_at_range_end"))

    return intervals, unknown_ranges, pl.DataFrame(anomalies)


state_machine_intervals, unknown_ranges, state_anomalies = analyze_state_machine(df, window_start, window_end)
summarize_intervals("state_machine", state_machine_intervals), summarize_intervals("unknown", unknown_ranges)


({'strategy': 'state_machine',
  'intervals': 645,
  'hours': 43.71908861111111,
  'min_seconds': 0.01,
  'median_seconds': 37.826,
  'max_seconds': 5103.793},
 {'strategy': 'unknown',
  'intervals': 1,
  'hours': 0.00015833333333333332,
  'min_seconds': 0.57,
  'median_seconds': 0.57,
  'max_seconds': 0.57})

## Interval-Intersection Consistency Check

This independently converts each state signal into intervals and intersects them. It should match the state-machine output unless there is a boundary or state-init bug.


In [7]:
def state_intervals(df: pl.DataFrame, on_event: str, off_event: str, window_start: datetime, window_end: datetime) -> tuple[list[Interval], list[dict[str, object]]]:
    intervals: list[Interval] = []
    anomalies: list[dict[str, object]] = []
    state: bool | None = None
    start: datetime | None = None
    start_line: int | None = None

    for row in df.filter(pl.col("timestamp") <= window_end).iter_rows(named=True):
        ts = row["timestamp"]
        event = row["event"]
        if event not in {on_event, off_event}:
            continue

        if event == on_event:
            if state is True:
                anomalies.append({"timestamp": ts, "line_number": row["line_number"], "event": event, "anomaly": f"repeated_{on_event.lower()}"})
                continue
            state = True
            start = max(ts, window_start)
            start_line = row["line_number"]
        else:
            if state is False:
                anomalies.append({"timestamp": ts, "line_number": row["line_number"], "event": event, "anomaly": f"repeated_{off_event.lower()}"})
                continue
            if state is True and start is not None:
                clipped = clip_interval(start, ts, window_start, window_end)
                if clipped is not None:
                    intervals.append(Interval(clipped[0], clipped[1], start_line, row["line_number"], f"{on_event}_to_{off_event}"))
            state = False
            start = None
            start_line = None

    if state is True and start is not None:
        clipped = clip_interval(start, window_end, window_start, window_end)
        if clipped is not None:
            intervals.append(Interval(clipped[0], clipped[1], start_line, None, "open_at_range_end"))
    return intervals, anomalies


def shutdown_gaps(df: pl.DataFrame, window_start: datetime, window_end: datetime) -> tuple[list[Interval], list[dict[str, object]]]:
    intervals: list[Interval] = []
    anomalies: list[dict[str, object]] = []
    in_shutdown = False
    start: datetime | None = None
    start_line: int | None = None

    for row in df.filter(pl.col("timestamp") <= window_end).iter_rows(named=True):
        ts = row["timestamp"]
        event = row["event"]
        if event == "DEVICE_SHUTDOWN":
            if in_shutdown:
                anomalies.append({"timestamp": ts, "line_number": row["line_number"], "event": event, "anomaly": "repeated_device_shutdown"})
                continue
            in_shutdown = True
            start = max(ts, window_start)
            start_line = row["line_number"]
        elif event == "DEVICE_STARTUP":
            if not in_shutdown:
                anomalies.append({"timestamp": ts, "line_number": row["line_number"], "event": event, "anomaly": "startup_without_shutdown"})
                continue
            if start is not None:
                clipped = clip_interval(start, ts, window_start, window_end)
                if clipped is not None:
                    intervals.append(Interval(clipped[0], clipped[1], start_line, row["line_number"], "shutdown_gap"))
            in_shutdown = False
            start = None
            start_line = None

    if in_shutdown and start is not None:
        clipped = clip_interval(start, window_end, window_start, window_end)
        if clipped is not None:
            intervals.append(Interval(clipped[0], clipped[1], start_line, None, "shutdown_gap_to_range_end"))
    return intervals, anomalies


def intersect_two(left: list[Interval], right: list[Interval], reason: str) -> list[Interval]:
    result = []
    i = j = 0
    left_sorted = sorted(left, key=lambda interval: (interval.start, interval.end))
    right_sorted = sorted(right, key=lambda interval: (interval.start, interval.end))
    while i < len(left_sorted) and j < len(right_sorted):
        a = left_sorted[i]
        b = right_sorted[j]
        start = max(a.start, b.start)
        end = min(a.end, b.end)
        if start < end:
            result.append(Interval(start, end, a.start_line, b.end_line, reason))
        if a.end <= b.end:
            i += 1
        else:
            j += 1
    return result


def subtract_intervals(base: list[Interval], blockers: list[Interval], reason: str) -> list[Interval]:
    result: list[Interval] = []
    for interval in base:
        pieces = [(interval.start, interval.end)]
        for blocker in blockers:
            next_pieces = []
            for start, end in pieces:
                if blocker.end <= start or blocker.start >= end:
                    next_pieces.append((start, end))
                    continue
                if start < blocker.start:
                    next_pieces.append((start, blocker.start))
                if blocker.end < end:
                    next_pieces.append((blocker.end, end))
            pieces = next_pieces
        result.extend(Interval(start, end, interval.start_line, interval.end_line, reason) for start, end in pieces if start < end)
    return result


screen_on, screen_anomalies = state_intervals(df, "SCREEN_INTERACTIVE", "SCREEN_NON_INTERACTIVE", window_start, window_end)
keyguard_hidden, keyguard_anomalies = state_intervals(df, "KEYGUARD_HIDDEN", "KEYGUARD_SHOWN", window_start, window_end)
shutdown_off, shutdown_anomalies = shutdown_gaps(df, window_start, window_end)

intersection_before_shutdown = intersect_two(screen_on, keyguard_hidden, "screen_and_keyguard")
intersection_intervals = subtract_intervals(intersection_before_shutdown, shutdown_off, "screen_and_keyguard_minus_shutdown")

summarize_intervals("interval_intersection", intersection_intervals)


{'strategy': 'interval_intersection',
 'intervals': 645,
 'hours': 43.71908861111111,
 'min_seconds': 0.01,
 'median_seconds': 37.826,
 'max_seconds': 5103.793}

## Comparison


In [8]:
summary = pl.DataFrame([
    summarize_intervals("existing_keyguard", existing_keyguard),
    summarize_intervals("existing_screen", existing_screen),
    summarize_intervals("new_state_machine", state_machine_intervals),
    summarize_intervals("new_interval_intersection", intersection_intervals),
    summarize_intervals("unknown_time", unknown_ranges),
])
summary


strategy,intervals,hours,min_seconds,median_seconds,max_seconds
str,i64,f64,f64,f64,f64
"""existing_keyguard""",634,46.150263,1.218,39.904,7488.602
"""existing_screen""",757,64.680914,0.0,27.372,25190.336
"""new_state_machine""",645,43.719089,0.01,37.826,5103.793
"""new_interval_intersection""",645,43.719089,0.01,37.826,5103.793
"""unknown_time""",1,0.000158,0.57,0.57,0.57


In [9]:
state_pairs = [(i.start, i.end) for i in state_machine_intervals]
intersection_pairs = [(i.start, i.end) for i in intersection_intervals]

consistency = {
    "state_machine_intervals": len(state_pairs),
    "intersection_intervals": len(intersection_pairs),
    "exact_match": state_pairs == intersection_pairs,
    "state_only": len(set(state_pairs) - set(intersection_pairs)),
    "intersection_only": len(set(intersection_pairs) - set(state_pairs)),
    "state_hours": sum((end - start).total_seconds() for start, end in state_pairs) / 3600,
    "intersection_hours": sum((end - start).total_seconds() for start, end in intersection_pairs) / 3600,
}
consistency


{'state_machine_intervals': 645,
 'intersection_intervals': 645,
 'exact_match': True,
 'state_only': 0,
 'intersection_only': 0,
 'state_hours': 43.71908861111111,
 'intersection_hours': 43.71908861111111}

In [10]:
if state_pairs != intersection_pairs:
    state_only = sorted(set(state_pairs) - set(intersection_pairs))[:20]
    intersection_only = sorted(set(intersection_pairs) - set(state_pairs))[:20]
    pl.DataFrame([
        {"side": "state_only", "start": start, "end": end, "seconds": (end - start).total_seconds()}
        for start, end in state_only
    ] + [
        {"side": "intersection_only", "start": start, "end": end, "seconds": (end - start).total_seconds()}
        for start, end in intersection_only
    ])
else:
    pl.DataFrame([{"message": "State-machine and interval-intersection outputs match exactly."}])


## Anomalies And Unknown Time


In [11]:
anomaly_frames = []
if not state_anomalies.is_empty():
    anomaly_frames.append(state_anomalies.with_columns(pl.lit("state_machine").alias("source")))
for source, rows in [
    ("screen_intervals", screen_anomalies),
    ("keyguard_intervals", keyguard_anomalies),
    ("shutdown_gaps", shutdown_anomalies),
]:
    if rows:
        anomaly_frames.append(pl.DataFrame(rows).with_columns(pl.lit(source).alias("source")))

all_anomalies = pl.concat(anomaly_frames, how="diagonal_relaxed") if anomaly_frames else pl.DataFrame()
if all_anomalies.is_empty():
    all_anomalies
else:
    all_anomalies.group_by(["source", "anomaly"]).agg(pl.len().alias("count")).sort(["source", "count"], descending=[False, True])


In [12]:
all_anomalies.sort(["timestamp", "line_number"]).head(50) if not all_anomalies.is_empty() else all_anomalies


timestamp,line_number,event,anomaly,previous_value,new_value,inside_window,source
datetime[μs],i64,str,str,bool,bool,bool,str
2026-07-26 20:37:10.915,43,"""KEYGUARD_SHOWN""","""repeated_keyguard_shown""",false,false,true,"""state_machine"""
2026-07-26 20:37:10.915,43,"""KEYGUARD_SHOWN""","""repeated_keyguard_shown""",null,null,null,"""keyguard_intervals"""
2026-07-26 20:37:16.646,47,"""SCREEN_NON_INTERACTIVE""","""repeated_screen_non_interactiv…",false,false,true,"""state_machine"""
2026-07-26 20:37:16.646,47,"""SCREEN_NON_INTERACTIVE""","""repeated_screen_non_interactiv…",null,null,null,"""screen_intervals"""
2026-07-29 23:49:00.795,1245,"""KEYGUARD_SHOWN""","""repeated_keyguard_shown""",false,false,true,"""state_machine"""
…,…,…,…,…,…,…,…
2026-08-01 23:41:53.376,2471,"""SCREEN_INTERACTIVE""","""repeated_screen_interactive""",null,null,null,"""screen_intervals"""
2026-08-01 23:42:04.858,2477,"""SCREEN_NON_INTERACTIVE""","""repeated_screen_non_interactiv…",false,false,true,"""state_machine"""
2026-08-01 23:42:04.858,2477,"""SCREEN_NON_INTERACTIVE""","""repeated_screen_non_interactiv…",null,null,null,"""screen_intervals"""


In [13]:
intervals_frame(unknown_ranges).sort("seconds", descending=True).head(20) if unknown_ranges else pl.DataFrame()


start,end,seconds,minutes,start_line,end_line,reason
datetime[μs],datetime[μs],f64,f64,null,i64,str
2026-07-26 18:15:27.403,2026-07-26 18:15:27.973,0.57,0.0095,null,2,"""state_unknown"""


## Longest New Active Intervals


In [ ]:
intervals_frame(state_machine_intervals).sort("seconds", descending=True).head(30)


## Shutdown Context


In [14]:
shutdown_lines = df.filter(pl.col("event").is_in(["DEVICE_SHUTDOWN", "DEVICE_STARTUP"])).select("line_number").to_series().to_list()
context_rows = []
for line_number in shutdown_lines:
    context_rows.append(df.filter((pl.col("line_number") >= line_number - 5) & (pl.col("line_number") <= line_number + 8)))
pl.concat(context_rows).unique(subset=["line_number"]).sort(["timestamp", "line_number"])


line_number,timestamp,event,event_type
i64,datetime[μs],str,i64
36,2026-07-26 19:45:03.478,"""KEYGUARD_SHOWN""",17
37,2026-07-26 19:45:09.214,"""SCREEN_INTERACTIVE""",15
38,2026-07-26 19:45:11.242,"""KEYGUARD_HIDDEN""",18
39,2026-07-26 19:45:30.581,"""SCREEN_NON_INTERACTIVE""",16
40,2026-07-26 19:45:32.174,"""KEYGUARD_SHOWN""",17
…,…,…,…
2722,2026-08-02 11:43:17.438,"""KEYGUARD_HIDDEN""",18
2723,2026-08-02 11:46:44.915,"""SCREEN_NON_INTERACTIVE""",16
2724,2026-08-02 11:46:45.441,"""KEYGUARD_SHOWN""",17
